In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Log Analytics Pipeline") \
    .getOrCreate()

print("Spark Started")

Spark Started


In [2]:
logs = spark.read.csv("../data/logs.csv", header=True, inferSchema=True)

logs.show(5)

+--------------------+---------+------+-------------+
|           timestamp|endpoints|status|response_time|
+--------------------+---------+------+-------------+
|2026-03-17 13:38:...| /profile|   200|          292|
|2026-03-17 13:38:...|    /cart|   200|          278|
|2026-03-17 13:38:...| /profile|   200|          386|
|2026-03-17 13:38:...|/checkout|   404|          121|
|2026-03-17 13:38:...|   /login|   500|          364|
+--------------------+---------+------+-------------+
only showing top 5 rows


In [3]:
from pyspark.sql.functions import col, avg, count

In [4]:
endpoints_stats = logs.groupBy("endpoints").agg(
    count("*").alias("total_requests"),
    avg("response_time").alias("avg_response_time")
)

endpoints_stats.show()

+---------+--------------+------------------+
|endpoints|total_requests| avg_response_time|
+---------+--------------+------------------+
|    /cart|         16583| 275.7915334981608|
|/products|         16684| 273.5463318149125|
|   /login|         16711| 273.9839626593262|
|/checkout|         16692| 276.7355020369039|
|  /orders|         16766|274.03829178098533|
| /profile|         16564| 275.9278555904371|
+---------+--------------+------------------+



In [5]:
error_stats = logs.filter(col("status") >= 400) \
    .groupBy("endpoints") \
    .count() \
    .withColumnRenamed("count", "error_count")

error_stats.show()

+---------+-----------+
|endpoints|error_count|
+---------+-----------+
|/products|       5645|
|    /cart|       5535|
|   /login|       5577|
|/checkout|       5544|
|  /orders|       5600|
| /profile|       5486|
+---------+-----------+



In [11]:
endpoints_stats.toPandas().to_csv("../data/endpoint_stats/endpoint_stats.csv", index=False)

error_stats.toPandas().to_csv("../data/error_stats/error_stats.csv", index=False)